# Exploração Inicial de Dados (EDA)
---

Com a etapa preliminar de limpeza concluída, iniciamos a exploração estrutural. O objetivo principal é avaliar os tipos de dados disponíveis, mapear a métrica volumétrica e identificar o nível de sanidade do *DataFrame*.

Carregamento inicial (`read_csv`):

In [3]:
import numpy as np
import pandas as pd

df = pd.read_csv("../../data/processed/olist_super_dataset.csv")

## Perfilamento Automatizado

O **YData Profiling** pode ser acionado para entregar uma varredura estática geral. Para compilar o relatório unificado localmente, execute o pipeline em `src/etl/data_exploration.py`.

Abaixo, realizamos uma inspeção estrita de metadados em Pandas, focando no consumo real de memória e na tipagem matricial (`df.info()`).

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109921 entries, 0 to 109920
Data columns (total 27 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       109921 non-null  object 
 1   customer_id                    109921 non-null  object 
 2   order_status                   109921 non-null  object 
 3   order_purchase_timestamp       109921 non-null  object 
 4   order_approved_at              109906 non-null  object 
 5   order_delivered_carrier_date   109920 non-null  object 
 6   order_delivered_customer_date  109921 non-null  object 
 7   order_estimated_delivery_date  109921 non-null  object 
 8   tempo_entrega_dias             109921 non-null  float64
 9   atraso_entrega                 109921 non-null  bool   
 10  customer_unique_id             109921 non-null  object 
 11  customer_zip_code_prefix       109921 non-null  int64  
 12  customer_city                 

## Volumetria e Consistência (Nulidade)

A matriz atual abriga mais de 100 mil registros, demandando operações estritamente vetorizadas e evitando loops imperativos.

Em seguida, acionamos verificações sobre a corrupção/ausência dos dados mapeando os valores `NaN` / `null`:

In [ ]:
tot_lines = len(df.index)
null_lines = df.isnull().any(axis=1).sum()

print(f"""Quantidade de linhas nulas: {null_lines}
Quantidade de células nulas: {df.isnull().sum().sum()}
Porcentagem em relação ao total: {(null_lines*100)/tot_lines:.2f}%""")

Quantidade de linhas nulas: 2360
Quantidade de células nulas: 5437
Porcentagem representada em relação ao total: 2.15%


### Tratamento de Nulos (Drop Otimizado)

Considerando que a taxa de corrupção equivale a **2.15%** do corpus, imputações matemáticas e estatísticas adicionariam complexidade com viés artificial, trazendo baixo ganho informacional para a análise atual. O descarte direto das linhas satisfaz o requisito de integridade com *zero overhead* lógico.

Abaixo, efetuamos a alteração referencial restrita à própria estrtura em memória (`inplace=True`):

In [9]:
df.dropna(axis=0, how='any', inplace=True)

tot_lines = len(df.index)
null_lines = df.isnull().any(axis=1).sum()

print(f"""Quantidade de linhas nulas: {null_lines}
Quantidade de células nulas: {df.isnull().sum().sum()}
Porcentagem em relação ao total: {(null_lines*100)/tot_lines:.2f}%""")

Quantidade de linhas nulas: 0
Quantidade de células nulas: 0
Porcentagem em relação ao total: 0.00%


## Verificação de Modelagem: Cardinalidade

Observe o comportamento da coluna `order_id`. Segundo a documentação da base (Olist/Kaggle), trata-se de um identificador único para os pedidos logísticos:

![Dicionário de Referência](./image/order_id_kagle.png)

Avaliamos o comportamento das entidades nas tabelas `originais` versus dataset `concatenado/mergeado` para expor o design de modelagem:

In [6]:
order_df = pd.read_csv("../../data/raw/olist_orders_dataset.csv")

print(f"""-> order_id's duplicados na table original: {order_df["order_id"].duplicated().sum()}
-> order_id's duplicados na Super Tabela (limpa): {df["order_id"].duplicated().sum()}""")

-> order_id's duplicados na table original: 0
-> order_id's duplicados na Super Tabela (limpa): 13688


### Conclusão Analítica: Desnormalização (1:N)

Conforme auditado, **não há inconsistência de limpeza**. A presença de IDs sequenciais repetidos na tabela limpa (`Super Tabela`) deriva de uma junção (`JOIN`/`MERGE`) com arquitetura relacional de cardinalidade **1:N** (Um Para Muitos).

A tabela raiz mantêm a singularidade do `order_id`, mas ao projetar os Itens (`order_items`), a mesma nota de compra se fragmente em *N* produtos diferentes. Cálculos logísticos ou de ticket-médio unificados daqui para frente exigem agregações restritivas explícitas (`df.groupby('order_id')`) para prevenir dupla contagem e inflação contábil dos KPIs.